# CFRI simple holdout v1 — yeni varlıklarda kilitli sınama

GitHub yüklemesini koşu tamamlandıktan sonra yapabilirsiniz. İlk **Run all**, gömülü kodun SHA-256 değerini ve UTC zamanını Google Drive'a kaydeder; veri indirme bu yerel dondurma kaydından sonra başlar. Bu kayıt dışarıdan doğrulanmış bir ön kayıt zaman damgası değildir. Protokol ve kapsam sınırları `PROTOCOL.md` içindedir.

Bu deney daha önce kullanılmamış 16 ETF'yi hedef alır; **2026 Ocak–Eylül dönemi aynı kaldığından yeni zaman dönemi doğrulaması değildir**. Tek tek alanlar ve tam satır kesintileri ayrı raporlanır. CFRI/SAITS aynı maskeleri kullanır; tam satır karşılaştırmasında ikisine de aynı peer köprüsü verilir. Özel Yahoo ham dosyaları sonuç ZIP'ine eklenmez.

Colab'da **Runtime > Change runtime type > GPU**, tercihen A100, seçin ve **Run all** deyin. Google Drive'a izin verin. Sonuçlar `MyDrive/ITRI/CFRI_holdout_v1_local` klasörüne koşul bazında kaydedilir. Oturum koparsa aynı notebook'ta yeniden **Run all** yapın; SHA ile doğrulanan koşullar atlanır, yalnız yarım kalan koşul tekrar hesaplanır. Son ZIP otomatik indirilir.


In [ ]:
import subprocess, sys
requirements = ['numpy==2.1.3', 'pandas==2.2.3', 'scikit-learn==1.7.2', 'yfinance==1.7.0', 'exchange-calendars==4.13.2', 'pypots==1.5']
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements])
print("Pinned Python packages installed; PyTorch comes from the selected Colab runtime.")


In [ ]:
import base64, hashlib, io, json, os, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
run_root = Path("/content/drive/MyDrive/ITRI/CFRI_holdout_v1_local")
run_root.mkdir(parents=True, exist_ok=True)
code_dir = Path("/content/cfri_holdout_v1_code")
code_dir.mkdir(parents=True, exist_ok=True)
packed = base64.b64decode('...')
embedded_bundle_sha256 = '8f7a5176ea48339980379ca1b802205d5a8b05e129ef96e9ef06c2a6ad776979'
assert hashlib.sha256(packed).hexdigest() == embedded_bundle_sha256
with zipfile.ZipFile(io.BytesIO(packed)) as z:
    if z.testzip() is not None:
        raise RuntimeError("Embedded code ZIP CRC failed")
    z.extractall(code_dir)
source_hashes = json.loads((code_dir / "SOURCE_SHA256.json").read_text())
for name, expected in source_hashes.items():
    if hashlib.sha256((code_dir / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Source hash mismatch: {name}")
print("Drive results:", run_root)
print("Embedded frozen source SHA-256:", embedded_bundle_sha256)


In [ ]:
source_path = run_root / "frozen_source_bundle.zip"
if source_path.exists():
    if hashlib.sha256(source_path.read_bytes()).hexdigest() != embedded_bundle_sha256:
        raise RuntimeError("Drive contains a different frozen source bundle; use the original notebook or a new root")
else:
    temp = source_path.with_name(source_path.name + ".tmp")
    temp.write_bytes(packed)
    os.replace(temp, source_path)
receipt_path = run_root / "freeze_receipt.json"
if receipt_path.exists():
    freeze_receipt = json.loads(receipt_path.read_text())
    if freeze_receipt.get("embedded_bundle_sha256") != embedded_bundle_sha256:
        raise RuntimeError("Freeze receipt changed or does not match this notebook")
else:
    freeze_receipt = {"embedded_bundle_sha256": embedded_bundle_sha256,
                      "freeze_utc": datetime.now(timezone.utc).isoformat()}
    temp = receipt_path.with_name(receipt_path.name + ".tmp")
    temp.write_text(json.dumps(freeze_receipt, sort_keys=True, indent=2) + "\n")
    os.replace(temp, receipt_path)
print("Freeze receipt:", freeze_receipt)


In [ ]:
import subprocess
from collections import deque

def run_stage(name, script, args):
    recent = deque(maxlen=60)
    command = [sys.executable, "-u", str(code_dir / script), "--root", str(run_root), *args]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          cwd=str(code_dir), text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            recent.append(line)
        status = process.wait()
    if status:
        detail = "".join(recent)
        (run_root / f"{name}_last_error.log").write_text(detail)
        raise RuntimeError(f"{name} stopped (exit {status}). Last output:\n{detail}")


In [ ]:
test_files = sorted(p.stem for p in code_dir.glob("test_holdout_*.py"))
subprocess.check_call([sys.executable, "-m", "unittest", "-q", *test_files], cwd=str(code_dir))
print("Synthetic tests passed:", test_files)


In [ ]:
run_stage("data", "holdout_run.py", ["--stage", "data"])


In [ ]:
run_stage("cpu", "holdout_run.py", ["--stage", "cpu"])


In [ ]:
saits_workers = None  # Automatic: up to 4 on an A100; 1 on smaller GPUs.
saits_args = [] if saits_workers is None else ["--workers", str(saits_workers)]
run_stage("saits", "holdout_saits.py", saits_args)


In [ ]:
from google.colab import files
run_stage("analyze", "holdout_analyze.py", [])
result_zip = run_root / "CFRI_holdout_v1_results.zip"
if not result_zip.is_file():
    raise FileNotFoundError(result_zip)
print("Completed ZIP:", result_zip)
files.download(str(result_zip))
